# Part 6: Noise and hardware

Practice notebook for *Qubit to Classifier*. Run the examples, fill in the exercises, and compare with the solutions at the end. Every cell needs only Python 3 and NumPy, except one optional Qiskit exercise in chapter 4.1.

## 6.1 Noise and mixed states

### Density matrices in NumPy

A density matrix is ρ = Σ p<sub>i</sub> |ψ<sub>i</sub>⟩⟨ψ<sub>i</sub>|, where |ψ⟩⟨ψ| is `np.outer(psi, psi.conj())`. The code builds the two mixtures from the chapter, checks that they are the same matrix, and reads off the Bloch vector and the purity.

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
s = 1 / np.sqrt(2)
ket0, ket1 = np.array([1, 0]), np.array([0, 1])
plus, minus = np.array([s, s]), np.array([s, -s])


def proj(psi):
    return np.outer(psi, psi.conj())             # |psi><psi|


def bloch(rho):
    return np.array([np.trace(rho @ P).real for P in (X, Y, Z)])


mix_z = 0.5 * proj(ket0) + 0.5 * proj(ket1)      # |0> or |1>, 50:50
mix_x = 0.5 * proj(plus) + 0.5 * proj(minus)     # |+> or |->, 50:50
print("same density matrix:", np.allclose(mix_z, mix_x))
for name, rho in [("mixture", mix_z), ("pure |+>", proj(plus))]:
    print(f"{name:9s} rho = {rho.real.round(3).tolist()}  Bloch = {bloch(rho).round(3) + 0}  purity = {np.trace(rho @ rho).real:.3f}")

The two recipes give the identical matrix I/2, with Bloch vector 0 and purity 1/2. The pure state |+⟩ has the same diagonal but off-diagonal coherences of 1/2, Bloch vector (1, 0, 0) and purity 1.

### Exercise 6.1.1: Noise channels with Kraus operators

Write `apply_channel(rho, kraus)`, which returns Σ K ρ K†. Apply a phase flip with p = 0.1 to |+⟩ ten times, and amplitude damping with γ = 0.3 to |+⟩ ten times, printing the Bloch vector every few steps. Where does each channel push the state?

<details><summary>Hint</summary>

Phase flip: K₀ = √(1 − p) I, K₁ = √p Z. Amplitude damping: K₀ = diag(1, √(1 − γ)), K₁ = [[0, √γ], [0, 0]].

</details>

In [ ]:
# Your code here

### Exercise 6.1.2: One qubit of an entangled pair is a mixture

For the state cos(θ/2)|00⟩ + sin(θ/2)|11⟩, compute the density matrix of q0 alone with the partial trace. Check that its Bloch vector is (0, 0, cos θ) for θ = π/3, and that the density matrix is I/2 for θ = π/2.

<details><summary>Hint</summary>

Reshape the four amplitudes into a 2 × 2 array a[q0, q1]. Then the state of q0 is `a @ a.conj().T`: summing over q1 is the partial trace.

</details>

In [ ]:
# Your code here

## 6.2 Decoherence and real hardware

### T1 and T2 decay

An idle qubit loses energy on the T1 clock and phase on the T2 clock. The code prints what is left of |+⟩ (its x coordinate) and of |1⟩ (its P(1)) over time, for T1 = 100 μs and T2 = 60 μs.

In [ ]:
import numpy as np

T1, T2 = 100.0, 60.0                       # microseconds
print(" time (us)   x of |+>   P(1) of |1>")
for t in [0, 10, 30, 60, 100, 200]:
    print(f"{t:9.0f}   {np.exp(-t / T2):9.3f}   {np.exp(-t / T1):10.3f}")

At 30 μs, x = e<sup>−0.5</sup> ≈ 0.61 and P(1) = e<sup>−0.3</sup> ≈ 0.74, the numbers of the chapter's worked example.

### Exercise 6.2.1: The error budget

Each two-qubit gate fails with probability ε. For ε = 0.5% and ε = 0.1%, print the chance that a circuit of N gates runs without any error, (1 − ε)<sup>N</sup>, next to the approximation e<sup>−Nε</sup>, for N = 100, 1,000 and 10,000. Then find the largest N with at least a 50% chance.

<details><summary>Hint</summary>

The 50% point is where Nε ≈ ln 2, so N ≈ ln 2 / ε.

</details>

In [ ]:
# Your code here

### Exercise 6.2.2: The repetition code, simulated

Encode a 0 as d copies, flip each copy with probability p, and decode by majority vote. For d = 3 and p = 1%, 10% and 40%, simulate 200,000 bits and compare the failure rate with 3p² − 2p³. Then try d = 5 and d = 7 at p = 10%.

<details><summary>Hint</summary>

`rng.random((M, d)) < p` gives M rows of d copies, True where a copy flipped. A row fails when more than half of its copies flipped.

</details>

In [ ]:
# Your code here

### Exercise 6.2.3: Correcting readout errors

Calibration shows that |0⟩ reads as 1 in 2% of shots and |1⟩ reads as 0 in 5% of shots. An experiment reads 1 in 40% of its shots. Write the calibration as a 2 × 2 matrix A, with observed probabilities = A × true probabilities, and invert it to find the true probability of 1.

<details><summary>Hint</summary>

Column 0 of A is what a true 0 looks like after readout, (0.98, 0.02), and column 1 is what a true 1 looks like, (0.05, 0.95). Solve with `np.linalg.solve(A, observed)`.

</details>

In [ ]:
# Your code here

---

# Solutions

### Solution 6.1.1: Noise channels with Kraus operators

In [ ]:
import numpy as np

I = np.eye(2)
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])


def apply_channel(rho, kraus):
    return sum(K @ rho @ K.conj().T for K in kraus)


def bloch(rho):
    return np.array([np.trace(rho @ P).real for P in (X, Y, Z)])


p, g = 0.1, 0.3
phase_flip = [np.sqrt(1 - p) * I, np.sqrt(p) * Z]
damping = [np.diag([1, np.sqrt(1 - g)]), np.array([[0, np.sqrt(g)], [0, 0]])]
plus = np.array([1, 1]) / np.sqrt(2)

for name, kraus in [("phase flip", phase_flip), ("amplitude damping", damping)]:
    rho = np.outer(plus, plus)
    print(name)
    for step in range(1, 11):
        rho = apply_channel(rho, kraus)
        if step in (1, 2, 5, 10):
            print(f"   after step {step:2d}: Bloch vector {bloch(rho).round(3) + 0}")

Expected output:

```
phase flip
   after step  1: Bloch vector [0.8 0.  0. ]
   after step  2: Bloch vector [0.64 0.   0.  ]
   after step  5: Bloch vector [0.328 0.    0.   ]
   after step 10: Bloch vector [0.107 0.    0.   ]
amplitude damping
   after step  1: Bloch vector [0.837 0.    0.3  ]
   after step  2: Bloch vector [0.7  0.   0.51]
   after step  5: Bloch vector [0.41  0.    0.832]
   after step 10: Bloch vector [0.168 0.    0.972]
```

The phase flip shrinks x toward 0 by a factor (1 − 2p) = 0.8 each time and leaves z alone: |+⟩ fades into the maximally mixed state. Amplitude damping pulls every state toward |0⟩, the north pole.

### Solution 6.1.2: One qubit of an entangled pair is a mixture

In [ ]:
import numpy as np

X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])

for theta in [np.pi / 3, np.pi / 2]:
    psi = np.array([np.cos(theta / 2), 0, 0, np.sin(theta / 2)])
    a = psi.reshape(2, 2)
    rho_q0 = a @ a.conj().T                         # partial trace over q1
    r = [np.trace(rho_q0 @ P).real for P in (X, Y, Z)]
    print(f"theta = {theta:.4f}: rho(q0) = {rho_q0.real.round(3).tolist()}  Bloch = {np.round(r, 3) + 0}  cos(theta) = {np.cos(theta) + 0:.3f}")

Expected output:

```
theta = 1.0472: rho(q0) = [[0.75, 0.0], [0.0, 0.25]]  Bloch = [0.  0.  0.5]  cos(theta) = 0.500
theta = 1.5708: rho(q0) = [[0.5, 0.0], [0.0, 0.5]]  Bloch = [0. 0. 0.]  cos(theta) = 0.000
```

### Solution 6.2.1: The error budget

In [ ]:
import numpy as np

for eps in [0.005, 0.001]:
    print(f"error rate {eps:.1%}:")
    for N in [100, 1000, 10_000]:
        print(f"   N = {N:6d}: (1 - eps)^N = {(1 - eps) ** N:.4f}   e^(-N eps) = {np.exp(-N * eps):.4f}")
    N_half = int(np.log(0.5) / np.log(1 - eps))
    print(f"   largest circuit with at least a 50% chance: {N_half} gates (ln 2 / eps = {np.log(2) / eps:.0f})")

Expected output:

```
error rate 0.5%:
   N =    100: (1 - eps)^N = 0.6058   e^(-N eps) = 0.6065
   N =   1000: (1 - eps)^N = 0.0067   e^(-N eps) = 0.0067
   N =  10000: (1 - eps)^N = 0.0000   e^(-N eps) = 0.0000
   largest circuit with at least a 50% chance: 138 gates (ln 2 / eps = 139)
error rate 0.1%:
   N =    100: (1 - eps)^N = 0.9048   e^(-N eps) = 0.9048
   N =   1000: (1 - eps)^N = 0.3677   e^(-N eps) = 0.3679
   N =  10000: (1 - eps)^N = 0.0000   e^(-N eps) = 0.0000
   largest circuit with at least a 50% chance: 692 gates (ln 2 / eps = 693)
```

### Solution 6.2.2: The repetition code, simulated

In [ ]:
import numpy as np

rng = np.random.default_rng(seed=10)
M = 200_000


def failure_rate(d, p):
    flips = rng.random((M, d)) < p
    return np.mean(flips.sum(axis=1) > d // 2)


for p in [0.01, 0.10, 0.40]:
    print(f"d = 3, p = {p:.2f}: simulated {failure_rate(3, p):.4f}   3p^2 - 2p^3 = {3 * p**2 - 2 * p**3:.4f}")
for d in [5, 7]:
    print(f"d = {d}, p = 0.10: simulated {failure_rate(d, 0.10):.4f}")

Expected output:

```
d = 3, p = 0.01: simulated 0.0003   3p^2 - 2p^3 = 0.0003
d = 3, p = 0.10: simulated 0.0282   3p^2 - 2p^3 = 0.0280
d = 3, p = 0.40: simulated 0.3536   3p^2 - 2p^3 = 0.3520
d = 5, p = 0.10: simulated 0.0087
d = 7, p = 0.10: simulated 0.0025
```

Below the threshold of 50% for this code, more copies make the logical error rarer. At p = 10%, three copies cut the error to 2.8%, five to 0.86% and seven to 0.27%. Those are the exact values. The simulated rates differ a little from them, and from run to run, because 200,000 trials still leave some sampling noise.

### Solution 6.2.3: Correcting readout errors

In [ ]:
import numpy as np

A = np.array([[0.98, 0.05],      # rows: read 0, read 1
              [0.02, 0.95]])     # columns: true 0, true 1
observed = np.array([0.60, 0.40])
true = np.linalg.solve(A, observed)
print("observed:", observed, "  corrected:", true.round(3))

Expected output:

```
observed: [0.6 0.4]   corrected: [0.591 0.409]
```

This reproduces the chapter's worked example, q ≈ 0.409. It corrects the statistics, never an individual shot.